In [0]:
df=spark.createDataFrame([(1,"ravi","hyderabad"),(2,"nallui","bangalore"),(3,"surya","chennai")],["cid","cname","clocation"])
df.show()


In [0]:
#write to parquet file
df.write.format("parquet").mode("overwrite").save("/Volumes/workspace/default/delta/parquet")



In [0]:
# write into delta file
# df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/delta/delta_new_12/")

# use select query on delta_new_12 table


In [0]:
#schema enforcement
df1=spark.createDataFrame([(1,"ravi","hyderabad","12345"),(2,"nallui","bangalore","12345"),(3,"surya","chennai","12345")],["cid","cname","clocation","contact"])
df1.show()

In [0]:
#write df1 into parquet file
df1.write.format("parquet").mode("overwrite").save("/Volumes/workspace/default/delta/parquet")

In [0]:
#write data into delta file
# df1.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/delta/delta_new/")

# since delta file previously having 3 columns. now we are trying to write data of 4 columns. hence we get schema mismatch error. if it is mandatory to write 4 columns (not the corrupted data), then we need to use "mergeschema option"
df1.write.format("delta").mode("overwrite").option("mergeschema",True).save("/Volumes/workspace/default/delta/delta_new_12/")


In [0]:
#read delta file data
spark.read.format("delta").load("/Volumes/workspace/default/delta/delta_new_12/").show()

In [0]:
#display current time
# spark.sql("select current_timestamp").show()
# df2=spark.createDataFrame([(1,"ravi","india","12345"),(2,"nallui","uk","12345"),(3,"surya","usa","12345")],["cid","cname","clocation","contact"])
# df2.show()

# # append new data to delta file
# df2.write.format("delta").mode("append").option("mergeschema",True).save("/Volumes/workspace/default/delta/delta_new_12/")

# #read delta file data
spark.read.format("delta").load("/Volumes/workspace/default/delta/delta_new_12/").show()




In [0]:
# read delta file data based on time travel
""" we have two options 1.versionAsOf 
                        2.timestampAsOf"""
spark.read.format("delta").load("/Volumes/workspace/default/delta/delta_new_12/",versionAsOf=2).show()
# spark.read.format("delta").load("/Volumes/workspace/default/delta/delta_new/",timestampAsOf="2026-03-09 12:37:21").show()

In [0]:
#create delta table
df3 = spark.read.format("delta").load("/Volumes/workspace/default/delta/delta_new_12/")

# Write as new delta table
df3.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.delta_table")
    

In [0]:
#read delta_table
spark.sql("select * from delta_table").show()



In [0]:
#update column value
# spark.sql("update delta_table set contact='6789' where cid='1'").show()
spark.sql("select * from delta_table version as of 6").show()

#recreate or replace delta table as of version 6
#spark.sql("create or replace table delta_table_1 as select * from delta_table version as of 6").show()
spark.sql("select * from delta_table_1").show()



In [0]:
#delete column where cid=2
# spark.sql("delete from delta_table where cid in('1','2','3')")
spark.sql("insert into delta_table values('4','anand','india','123457896')")
spark.sql("insert into delta_table values('1','anand','india','123457896')")
spark.sql("select * from delta_table").show()

In [0]:
#merge delata_table table into delta_table_1 table
spark.sql("MERGE INTO delta_table_1 AS t USING delta_table AS s ON t.cid = s.cid WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
# select delta_table_1
spark.sql("select * from delta_table_1").show()

## describe table commands in sql mode and pyspark mode

In [0]:
%sql
-- describe table in sql mode
-- describe extended delta_table_1
-- describe history delta_table_1
describe table in dsl
spark.sql(""" describe extended delta_table_1""").show()

### Deletion vector 

In [0]:
%sql
select * from delta_table_1